**Check Hong Kong Cantonese Corpus**

In [ ]:
import polars as pl 
import pycantonese
import warnings

warnings.filterwarnings('ignore')

corpus = pycantonese.hkcancor()
df = pl.read_csv("./output/filtered_dictionary_senses.csv")
skipped = list()
for row in df.iter_rows(named=True):
    jyutping = row["jyutping"]
    # Remove spaces
    jyutping = "".join(jyutping.split())
    results = corpus.search(character=row["traditional"], by_utterances=True)
    if not results:
        # Skip for now
        skipped.append(row)


**Load dataset and convert to traditional characters**

In [ ]:
from hanziconv import HanziConv
import polars as pl 

df = pl.read_csv("./output/Filtered Dictionary Senses - cleaned.csv")

df = df.with_columns([
    pl.col("traditional").map_elements(
        lambda x: HanziConv.toTraditional(x) if x is not None else None,
        return_dtype=pl.Utf8
    ),
    pl.col("sentence1").map_elements(
        lambda x: HanziConv.toTraditional(x) if x is not None else None,
        return_dtype=pl.Utf8
    ),
    pl.col("sentence2").map_elements(
        lambda x: HanziConv.toTraditional(x) if x is not None else None,
        return_dtype=pl.Utf8
    )
])

for row in df.iter_rows(named=True):
    assert row["traditional"] in row["sentence1"]
    assert row["traditional"] in row["sentence2"]

# df.write_csv("./output/filtered_dictionary_senses-traditional.csv")


**Run WSD**

In [29]:
import torch
from transformers import BertForMaskedLM, AlbertTokenizer, BertTokenizer

model_options = {
    "yue-scratch": "./yue-scratch",
    "bert-base-chinese": "google-bert/bert-base-chinese",
    "bert-base-cantonese": "indiejoseph/bert-base-cantonese"
}
# NOTE: Update model path before running
model_path = model_options["yue-scratch"]

if "yue-scratch" in model_path:
    tokenizer_class = AlbertTokenizer
else:
    tokenizer_class = BertTokenizer

model = BertForMaskedLM.from_pretrained(model_path)
tokenizer = tokenizer_class.from_pretrained(model_path)

In [ ]:
import polars as pl 
from itertools import combinations
from torch import cosine_similarity
from termcolor import colored 

df = pl.read_csv("./output/filtered_dictionary_senses-traditional.csv")

total_pairs = 0

predictions = []

def get_embedding(sentence, word, tokenizer, model):
    """Extract BERT embedding for a masked word in a sentence."""
    sentence = sentence.replace(word, "[MASK]")
    input_ids = tokenizer.encode(sentence, return_tensors='pt')
    
    mask_token_indices = torch.where(input_ids == tokenizer.mask_token_id)[1]
    mask_token_index = mask_token_indices[0].item()
    
    with torch.no_grad():
        outputs = model(input_ids, output_hidden_states=True)
    
    last_hidden_state = outputs.hidden_states[-1]
    mask_embedding = last_hidden_state[0, mask_token_index, :]
    
    return mask_embedding


for word, group in df.with_row_index().group_by("traditional"):
    # Step 1: Create all embeddings for this word group
    embeddings_cache = {}  # {row_index: (embedding1, embedding2)}
    
    for row in group.iter_rows(named=True):
        idx = row["index"]
        sent1 = row["sentence1"]
        sent2 = row["sentence2"]
        
        emb1 = get_embedding(sent1, word[0], tokenizer, model)
        emb2 = get_embedding(sent2, word[0], tokenizer, model)
        
        embeddings_cache[idx] = (emb1, emb2)
    
    # Step 2: Generate combinations and compare
    indices = group.select("index").to_series().to_list()

    for index1, index2 in combinations(indices, r=2):
        # Get precomputed embeddings
        def1_emb1, def1_emb2 = embeddings_cache[index1]
        def2_emb1, def2_emb2 = embeddings_cache[index2]
        
        # Calculate similarities
        same1 = cosine_similarity(def1_emb1, def1_emb2, dim=0)
        same2 = cosine_similarity(def2_emb1, def2_emb2, dim=0)
        
        diff1 = cosine_similarity(def1_emb1, def2_emb1, dim=0)
        diff2 = cosine_similarity(def1_emb2, def2_emb2, dim=0)
        
        averaged_same_score = (same1 + same2) / 2
        averaged_diff_score = (diff1 + diff2) / 2
        
        if averaged_same_score > averaged_diff_score:
            predictions.append("right")
        else:
            predictions.append("wrong")
            # Get original sentences for debugging
            def1_row = df.row(index1, named=True)
            def2_row = df.row(index2, named=True)
            sentences = (def1_row["sentence1"], def1_row["sentence2"],
                   def2_row["sentence1"], def2_row["sentence2"])
            sentences = [s.replace(word[0], f"_[{word[0]}]_") for s in sentences]
            print(sentences)

print(predictions)

['張糖紙唔食得㗎，快啲_[𦧲]_返齣嚟啦！', '食魚要_[𦧲]_骨。', '細佬成日都_[𦧲]_我去同佢食雪糕。', '攞份錶都要俾黑錢，死_[𦧲]_難_[𦧲]_先_[𦧲]_入個交易場。']
['我對眼就係法庭，你哋而傢_[啹]_唔_[啹]_先？', '我傢姐上年年尾俾人無理炒魷，覺得好唔_[啹]_。', '佢唔_[啹]_我好耐啦，今次公報私仇咋嘛！', '佢哋兩個都唔_[啹]_對方，開親會都鬧交，點閤作？']
['我對眼就係法庭，你哋而傢_[啹]_唔_[啹]_先？', '我傢姐上年年尾俾人無理炒魷，覺得好唔_[啹]_。', '頭先搬嘢又熱又攰，飲完杯嘢成個人_[啹]_曬！', '爸爸锡咗佢兩啖，佢個心都_[啹]_曬啦。']
['細路仔唔準車大_[炮]_㗎。', '噉嘅大_[炮]_你都信？', '唔好咁大_[炮]_咯。', '我講嘅唔係大_[炮]_，真㗎。']
['_[𢺳]_住塊大石爬上去。', '你_[𢺳]_住個扶手就唔驚啦。', '下半場一定要將啲分_[𢺳]_翻。', '最後又_[𢺳]_翻幾分？']
['上便_[撠]_住，你褪落啲就過得嘞。', '嗰條棍_[撠]_實喺裏頭，拎唔齣。', '你伸隻腳齣嚟想_[撠]_死人咩！', '佢_[撠]_咗嚇，就躀低咗。']
['呢個人都幾惡_[撠]_㗎。', '你份人真係好_[撠]_啊，一啲都唔識變通嘅。', '你伸隻腳齣嚟想_[撠]_死人咩！', '佢_[撠]_咗嚇，就躀低咗。']
['唔好嬲啦，嗰啲人係噉_[㗎啦]_。', '最好係各自寫平安紙，寫清楚啲就得_[㗎啦]_。', '我哋窮人都係天生天養_[㗎啦]_。', '睇死你係噉_[㗎啦]_！']
['唔應該_[嘥]_人哋。', '畀人當麵_[嘥]_。', '個名師講座你冇去聽真係_[嘥]_曬。', '咁好嘅機會_[嘥]_咗。']
['食埋個麪包，唔好_[嘥]_嘢呀。', '_[嘥]_咗好多時間。', '個名師講座你冇去聽真係_[嘥]_曬。', '咁好嘅機會_[嘥]_咗。']
['呢個國傢_[禁]_咗香口膠㗎。', '乜而傢中學已經冇左髮_[禁]_？', '十年咁耐，_[禁]_等咯。', '貴利唔能夠藉㗎，利遝利，好_[禁]_還㗎。']
['呢個國傢_[禁]_咗香口膠㗎。', '乜而傢中學已經冇左髮_[禁]_？', '佢又畀屋企人_[禁]_足，冇得齣街玩。', '我

In [32]:
from collections import Counter 

result_counter = Counter(predictions)
result_counter

Counter({'right': 71, 'wrong': 38})

In [33]:
result_counter["right"] / (result_counter["right"] + result_counter["wrong"])

0.6513761467889908